In [8]:
import pandas as pd
from evaluation import match, classify_by_thresh


def evaluation(
    lang: str, feature_cols: list[str], weights: list[float], threshold: float
):
    df = pd.read_csv(f"../benchmark/ground_truth/{lang}_ground_truth.csv")
    phantom_pairs = pd.read_csv(
        f"../benchmark/ground_truth/{lang}_validation_results.csv"
    )
    phantom_pairs = phantom_pairs[~phantom_pairs["validation"]][
        ["record_id", "old_index", "new_index"]
    ]
    df["score"] = sum(
        df[feature_cols[i]] * weights[i] for i in range(len(feature_cols))
    )
    df["index_sim"] = abs(df["old_index"] - df["new_index"])
    matched_df = match(df)
    final_res = classify_by_thresh(matched_df, threshold, phantom_pairs)
    final_res.to_excel(
        f"../benchmark/evaluation/{lang}_evaluation_results.xlsx", index=False
    )
    num_records = final_res["record_id"].nunique()
    num_mappings = len(final_res[final_res["label"] == 1])
    num_preds = len(final_res[final_res["pred"] == 1])
    num_TP = len(final_res[(final_res["label"] == 1) & (final_res["pred"] == 1)])
    num_FP = len(final_res[(final_res["label"] == 0) & (final_res["pred"] == 1)])
    num_FN = len(final_res[(final_res["label"] == 1) & (final_res["pred"] == 0)])
    precision = num_TP / num_preds
    recall = num_TP / num_mappings
    F1 = 2 * precision * recall / (precision + recall)
    return [
        num_records,
        num_mappings,
        num_preds,
        num_TP,
        num_FP,
        num_FN,
        precision * 100,
        recall * 100,
        F1 * 100,
    ]


java_res = evaluation(
    "java", ["class_sim", "method_sim", "arg_sim"], [1 / 3, 1 / 3, 1 / 3], 0.45
)
py_res = evaluation("py", ["fqn_sim", "arg_sim"], [1 / 2, 1 / 2], 0.35)
pd.DataFrame(
    [["Java"] + java_res, ["Python"] + py_res],
    columns=[
        "Language",
        "Records",
        "GT",
        "Predicted",
        "TP",
        "FP",
        "FN",
        "Precision",
        "Recall",
        "F1 score",
    ],
)

,Language,Records,GT,Predicted,TP,FP,FN,Precision,Recall,F1 score
0,Java,638,344,333,305,28,39,91.591592,88.662791,90.103397
1,Python,609,242,172,155,17,87,90.116279,64.049587,74.879227


In [9]:
import pandas as pd


def fp_analysis():
    for lang in ["java", "py"]:
        phantom_pairs = pd.read_csv(
            f"../benchmark/ground_truth/{lang}_validation_results.csv"
        )
        df = pd.read_excel(f"../benchmark/evaluation/{lang}_evaluation_results.xlsx")
        fp = df[(df["label"] == 0) & (df["pred"] == 1)][
            ["record_id", "old_index", "new_index"]
        ].merge(phantom_pairs)
        num_removal = len(fp[fp["old_removal"]])
        num_depre = len(fp[fp["old_deprecated"]])
        num_fp = len(fp)
        print(f"{lang}: {num_fp} false positives")
        print(f"    removal: {num_removal} ({num_removal/num_fp*100:.1f}%)")
        print(f"    deprecated: {num_depre} ({num_depre/num_fp*100:.1f}%)")


fp_analysis()

java: 28 false positives
    removal: 24 (85.7%)
    deprecated: 4 (14.3%)
py: 17 false positives
    removal: 4 (23.5%)
    deprecated: 13 (76.5%)


In [10]:
def fn_analysis():
    thresh = {"java": 0.45, "py": 0.35}
    for lang in ["java", "py"]:
        validation = pd.read_csv(
            f"../benchmark/ground_truth/{lang}_validation_results.csv"
        )
        df = pd.read_excel(f"../benchmark/evaluation/{lang}_evaluation_results.xlsx")
        fn = df[(df["label"] == 1) & (df["pred"] == 0)].merge(validation)
        num_fn = len(fn)
        print(f"{lang}: {num_fn} false negatives")
        num_low_sim = len(fn[fn["score"] <= thresh[lang]])
        print(f"    low similarity: {num_low_sim} ({num_low_sim/num_fn*100:.1f}%)")
        fn_phase4 = fn[fn["score"] > thresh[lang]]
        num_non_exist = len(
            fn_phase4[~(fn_phase4["old_existence"] & fn_phase4["new_existence"])]
        )
        print(f"    API not exist: {num_non_exist} ({num_non_exist/num_fn*100:.1f}%)")
        num_no_depre = len(
            fn_phase4[fn_phase4["old_existence"] & fn_phase4["new_existence"]]
        )
        print(f"    No deprecation: {num_no_depre} ({num_no_depre/num_fn*100:.1f}%)")


fn_analysis()

java: 39 false negatives
    low similarity: 16 (41.0%)
    API not exist: 22 (56.4%)
    No deprecation: 1 (2.6%)
py: 87 false negatives
    low similarity: 22 (25.3%)
    API not exist: 51 (58.6%)
    No deprecation: 14 (16.1%)


In [11]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]


def col2df(lang: str):
    data = []
    col = db[f"{lang}_candidate_update_instances"]
    for doc in col.find(projection={"_id": 0}):
        res = [doc["commit"], doc["library"]]

        version_before = doc["version_before"]
        version_after = doc["version_after"]
        old_api = doc["old_callee"]["full_name"]
        new_api = doc["new_callee"]["full_name"]
        if lang == "java":
            old_api = f"{old_api}({','.join(doc['most_prob_old_param_types'])})"
            new_api = f"{new_api}({','.join(doc['most_prob_new_param_types'])})"
        parts1 = [int(_) for _ in version_before.split(".")]
        parts2 = [int(_) for _ in version_after.split(".")]
        if parts1 > parts2:
            res.extend(
                [
                    version_after,
                    version_before,
                    True,
                    new_api,
                    old_api,
                    doc["new_existence"],
                    doc["old_existence"],
                ]
            )
        else:
            res.extend(
                [
                    version_before,
                    version_after,
                    False,
                    old_api,
                    new_api,
                    doc["old_existence"],
                    doc["new_existence"],
                ]
            )

        res.extend([doc["old_removal"], doc["old_deprecated"], doc["validation"]])
        data.append(res)
    df = pd.DataFrame(
        data,
        columns=[
            "commit",
            "library",
            "old_version",
            "new_version",
            "downgrade",
            "old_api",
            "new_api",
            "old_existence",
            "new_existence",
            "old_removal",
            "old_deprecated",
            "validation",
        ],
    )
    df.to_csv(f"../benchmark/final/{lang}_dataset_metadata.csv", index=False)
    return df


dfs = {"java": col2df("java"), "python": col2df("py")}

In [12]:
def scale_analysis():
    data = []
    for lang, df in dfs.items():
        df = df[df["validation"]]
        num_instances = len(df)
        num_commits = df["commit"].nunique()
        num_libs = df["library"].nunique()
        num_lib_versions = pd.concat(
            [
                df["library"] + " " + df["old_version"],
                df["library"] + " " + df["new_version"],
            ]
        ).nunique()
        num_version_updates = len(
            df.drop_duplicates(subset=["library", "old_version", "new_version"])
        )
        num_mappings = len(df.drop_duplicates(subset=["library", "old_api", "new_api"]))
        num_pairs = len(
            df.drop_duplicates(
                subset=["library", "old_version", "new_version", "old_api", "new_api"]
            )
        )
        data.append(
            [
                lang,
                num_libs,
                num_lib_versions,
                num_version_updates,
                num_mappings,
                num_pairs,
                num_instances,
                num_commits,
            ]
        )
    return pd.DataFrame(
        data,
        columns=[
            "Language",
            "Libraries",
            "Releases",
            "Version Transitions",
            "Mappings",
            "Pairs",
            "Instances",
            "Commits",
        ],
    )


scale_analysis()

,Language,Libraries,Releases,Version Transitions,Mappings,Pairs,Instances,Commits
0,java,2557,12165,10637,18900,35532,381661,17266
1,python,999,4908,5190,4456,11393,277259,7850


In [13]:
def mine_process():
    for lang, df in dfs.items():
        num_candidate = len(df)
        num_final = len(df[df["validation"]])
        print(f"{lang}: {num_candidate} candidate API update instances")
        non_existence = len(df[~(df["old_existence"] & df["new_existence"])])
        print(f"    {non_existence} API signature resolution failures")
        non_deprecate = num_candidate - num_final - non_existence
        print(f"    {non_deprecate} no deprecation or removal evidences")
        print(f"    {num_final} validated API update instances")
        num_rmv_path = len(df[df["validation"] & df["old_removal"]])
        print(f"    {num_rmv_path} passes API removal path")
        num_deprec_path = num_final - num_rmv_path
        print(f"    {num_deprec_path} passes API deprecation path")


mine_process()

java: 686608 candidate API update instances
    238403 API signature resolution failures
    66544 no deprecation or removal evidences
    381661 validated API update instances
    111925 passes API removal path
    269736 passes API deprecation path
python: 561018 candidate API update instances
    85926 API signature resolution failures
    197833 no deprecation or removal evidences
    277259 validated API update instances
    228840 passes API removal path
    48419 passes API deprecation path
